# Trust Analysis of Traffic Sign Classifiers - Training & Stress Suite

This notebook provides the complete pipeline for training and evaluating Traffic Sign Classifiers for the TSR Thesis project. It is optimized for Google Colab with GPU acceleration.

### Pipeline Overview:
1. **Environment Setup**: Repository cloning and dependency installation.
2. **Data Preparation**: Downloading and restructuring the GTSRB dataset.
3. **Model Training**: Running the training loop with W&B logging.
4. **Uncertainty Stress Suite**: Evaluating robustness against noise, blur, and weather corruptions.
5. **Result Export**: Automated downloading of metrics and model weights.

## 1. Environment Setup
Cloning the latest version of the repository and installing scientific dependencies.

In [ ]:
# Clean up if the folder exists to ensure a fresh clone
!rm -rf tsr-thesis

# Clone the repository
!git clone https://github.com/ucefhesham/tsr-thesis.git
%cd tsr-thesis

# Install dependencies (quietly)
!pip install -r requirements.txt -q
!pip install wandb -U -q

print("\n--- Setup Complete ---")

## 2. Weights & Biases Logging
Login is required to sync your training progress and model checkpoints.

In [ ]:
import wandb
wandb.login()

## 3. Storage & Checkpoint Management

### Option A: Mount Google Drive (Recommended for Persistence)
Mounting logic allows you to save checkpoints directly to your Drive so they aren't lost when the Colab instance times out.

In [ ]:
from google.colab import drive
import os

# Mount drive
drive.mount('/content/drive')

# Optional: Link your local checkpoints folder to Drive for automatic saving
# !ln -s /content/drive/MyDrive/thesis_checkpoints ./checkpoints

### Option B: Manual Upload (Local -> Colab)
Run this cell to upload a checkpoint from your computer (e.g., `epoch_017.ckpt`) to the current Colab environment.

In [ ]:
from google.colab import files
import os

os.makedirs('checkpoints', exist_ok=True)
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, os.path.join('checkpoints', filename))
    print(f"Moved {filename} to checkpoints/")

## 4. Data Preparation
GTSRB is a large dataset (~300MB). This cell ensures all partitions (Train, Test, and Test Ground Truth) are correctly downloaded into the Colab environment.

In [ ]:
from src.datamodules.gtsrb_module import GTSRBDataModule

dm = GTSRBDataModule(data_dir="data/")
dm.prepare_data()
print("\n--- GTSRB Dataset Ready ---")

## 5. Model Training
You can train the standard **ResNet-18 Baseline** or the **Evidential Model** by passing the appropriate model config.

In [ ]:
# OPTION A: Train ResNet-18 Baseline
!python train.py model=resnet18 trainer.max_epochs=20 trainer.devices=1 trainer.accelerator='gpu'

# OPTION B: Train Evidential Model (uncomment below)
# !python train.py model=evidential trainer.max_epochs=20 trainer.devices=1 trainer.accelerator='gpu'

## 6. Uncertainty Stress Suite
Evaluate robustness across 4 corruption categories and 5 severity levels. This script automatically performs temperature scaling and exports a final CSV for analysis.

In [ ]:
# Provide the path to your best checkpoint (trained in step 4 or uploaded in step 3)
CHECKPOINT_PATH = "checkpoints/epoch_017.ckpt"  # Change this to your preferred checkpoint

# Run the automated Stress Suite sweep
# Note: The script finds the GPU automatically!
!python eval.py ckpt_path={CHECKPOINT_PATH}

print("\n--- Evaluation Complete ---")

## 7. Export Results
Download the metrics CSV and the model weights back to your local machine.

In [ ]:
from google.colab import files
import os
import glob

# 1. Download ALL Log Results (CSVs, PNGs)
# This ensures you get both 'resnet_stress_test_results.csv' and any 'worst_offenders.png'
log_files = glob.glob('logs/*')
for f in log_files:
    if os.path.isfile(f):
        print(f"Downloading {f}...")
        files.download(f)

# 2. Download Model Checkpoints
ckpt_files = glob.glob('checkpoints/*.ckpt')
if ckpt_files:
    # Finds the newest .ckpt file created during this session
    latest_ckpt = max(ckpt_files, key=os.path.getctime)
    print(f"Downloading {latest_ckpt}...")
    files.download(latest_ckpt)